In [32]:
import pandas as pd
import numpy as np
import os
import glob
import sys

In [33]:
#path to folder with all subjects data

path = '/media/hcs-sci-psy-narun/OREGON_ADHD_1000/OREGON_ONE_SINGLE/derivatives/freesurfer/'
path_out = '/media/hcs-sci-psy-narun/Jack/Oregon_Structural/'
if not os.path.exists(path_out):
    os.makedirs(path_out)

In [34]:
#Freesurfer header for indexing

#indexes for cortex
fs_cort_ind = np.loadtxt('/media/hcs-sci-psy-narun/BackupDataD800/Alina/atlases/destrieux2009_new_header_for_table_WITHOUT_med_wall.txt', 
                         dtype=str)
#indexes for subcortex
fs_subc_ind = [
    'Left-Accumbens-area',
    'Right-Accumbens-area',
    'Left-Amygdala',
    'Right-Amygdala',
    'Brain-Stem',
    'Left-Caudate',
    'Right-Caudate',
    'Left-Cerebellum-Cortex',
    'Right-Cerebellum-Cortex',
    'Left-VentralDC',
    'Right-VentralDC',
    'Left-Hippocampus',
    'Right-Hippocampus',
    'Left-Pallidum',
    'Right-Pallidum',
    'Left-Putamen',
    'Right-Putamen',
    'Left-Thalamus-Proper',
    'Right-Thalamus-Proper'
]
#indexes for subcortex like in wb_commands
fs_subc_ind_wb = np.loadtxt('/media/hcs-sci-psy-narun/BackupDataD800/Alina/atlases/fs_index_subc.txt', dtype=str)

#total brain volume index
tot_names = [
    'Estimated Total Intracranial Volume',
    'Total cortical gray matter volume',
    'Subcortical gray matter volume',
    'Total cerebral white matter volume',
    'Ratio of BrainSegVol to eTIV'
]

tot_names_ml = [
    'FS_IntraCranial_Vol',
    'FS_TotCort_GM_Vol',
    'FS_SubCort_GM_Vol',
    'FS_Tot_WM_Vol',
    'FS_BrainSegVol_eTIV_Ratio'    
]

/tmp/ipykernel_2636268/4124026098.py:4: UserWarning: Input line 149 contained no data and will not be counted towards `max_rows=50000`.  This differs from the behaviour in NumPy <=1.22 which counted lines rather than rows.  If desired, the previous behaviour can be achieved by using `itertools.islice`.
Please see the 1.23 release notes for an example on how to do this.  If you wish to ignore this warning, use `warnings.filterwarnings`.  This warning is expected to be removed in the future and is given only once per `loadtxt` call.
  fs_cort_ind = np.loadtxt('/media/hcs-sci-psy-narun/BackupDataD800/Alina/atlases/destrieux2009_new_header_for_table_WITHOUT_med_wall.txt',


In [35]:
#read subjects ids with full directory names
subjects = []
for file in os.listdir(path):
    if os.path.isdir(os.path.join(path, file)) and file.startswith('sub-NDAR'):
        subjects.append(file)  # Store the full directory name

# Create a dictionary to track subjects and their timepoints
subject_timepoints = {}
for folder in os.listdir(path):
    if os.path.isdir(os.path.join(path, folder)) and folder.startswith('sub-NDAR'):
        # Extract the subject ID
        sub_id = folder.split('Age')[0].replace('sub-', '')

        # Extract the study year (timepoint)
        if 'StudyYear' in folder:
            timepoint = folder.split('StudyYear')[1][:2]  # Get the 2-digit year
        else:
            timepoint = 'unknown'

        # Add to tracking dict
        if sub_id not in subject_timepoints:
            subject_timepoints[sub_id] = []

        subject_timepoints[sub_id].append({
            'timepoint': timepoint,
            'full_dirname': folder
        })

In [36]:
pd.DataFrame(subjects).to_csv(path_out + 'st_subject_list.tsv', sep='\t')

In [37]:
#name of files to convert
files = ['lh.aparc.a2009s.stats', 'rh.aparc.a2009s.stats', 'aseg.stats']

In [38]:
#convert FreeSurfer table into csv-files for each subject
dct1={}
dct2={}
dct3={}

for subject_dir in subjects:
    for filename in files:
        # Adjust this path to match your actual FreeSurfer directory structure
        file = os.path.join(path, subject_dir, 'stats', filename)

        if os.path.isfile(file):
            #read table
            table = pd.DataFrame(np.loadtxt(file, dtype=str))
            #read headers
            header_names = [l.replace('\n', '').replace('  ', ' ').replace('# ', '').split(' ')[1:(len(table.columns)+1)] for l in open(file).readlines() if 'ColHeaders' in l]
            #rename table
            table.columns = header_names[0]
            table.index = table['StructName']
            table = table.drop('StructName', axis=1)
            #save table to dct
            if filename == 'lh.aparc.a2009s.stats':
                dct1[subject_dir]=table
            elif filename == 'rh.aparc.a2009s.stats':
                dct2[subject_dir]=table
            else:
                dct3[subject_dir]=table

        else:
            print('does not exist')
            print(file)

/tmp/ipykernel_2636268/86950943.py:13: UserWarning: Input line 1 contained no data and will not be counted towards `max_rows=50000`.  This differs from the behaviour in NumPy <=1.22 which counted lines rather than rows.  If desired, the previous behaviour can be achieved by using `itertools.islice`.
Please see the 1.23 release notes for an example on how to do this.  If you wish to ignore this warning, use `warnings.filterwarnings`.  This warning is expected to be removed in the future and is given only once per `loadtxt` call.
  table = pd.DataFrame(np.loadtxt(file, dtype=str))
/tmp/ipykernel_2636268/86950943.py:13: UserWarning: Input line 1 contained no data and will not be counted towards `max_rows=50000`.  This differs from the behaviour in NumPy <=1.22 which counted lines rather than rows.  If desired, the previous behaviour can be achieved by using `itertools.islice`.
Please see the 1.23 release notes for an example on how to do this.  If you wish to ignore this warning, use `war

In [39]:
#assemble total brain volumes into separate dictionary
dct_tvol = {}
for subject_dir in subjects:
    try:
        # Direct path to the aseg.stats file
        file_aseg = open(os.path.join(path, subject_dir, 'stats', files[2])).readlines()
        tvolvalues = []
        for tot in tot_names:
            for line in file_aseg:
                if tot in line:
                    tvolvalues+=[line.split(',')[-2]]
        dct_tvol[subject_dir] = pd.Series(tvolvalues, index=tot_names_ml)
    except FileNotFoundError:
        print(f"File not found: {os.path.join(path, subject_dir, 'stats', files[2])}")
        continue

In [40]:
#assemble table by modalities

dct_thick = {}
dct_area = {}
dct_subc = {}

for subject_dir in subjects:
    # Check if this subject has all the necessary data
    if subject_dir not in dct1 or subject_dir not in dct2 or subject_dir not in dct3:
        print(f"Missing data for {subject_dir}, skipping...")
        continue

    #load individual files
    df1 = dct1[subject_dir]
    df2 = dct2[subject_dir]
    df3 = dct3[subject_dir]

    #combine lh and rh into one vector
    thck_full = np.concatenate([df1['ThickAvg'], df2['ThickAvg']], axis=0)
    area_full = np.concatenate([df1['SurfArea'], df2['SurfArea']], axis=0)

    #filter to chosen structures
    try:
        df3 = df3.reindex(index=fs_subc_ind)

        #write to dictionary
        dct_thick[subject_dir] = thck_full
        dct_area[subject_dir] = area_full
        dct_subc[subject_dir] = df3['Volume_mm3']
    except KeyError as e:
        print(f"Error processing {subject_dir}: {e}")
        continue

In [41]:
#check if something is too short
todel = []

print('thickness')
for key in dct_thick.keys():
    if len(dct_thick[key]) < 148:
        print('short  ', key, "  it's length is", len(dct_thick[key]))
        todel += [key]

print('areas')        
for key in dct_area.keys():
    if len(dct_area[key]) < 148:
        print('short  ', key, "  it's length is", len(dct_area[key]))
        todel += [key]
        
print('subcortex')
for key in dct_subc.keys():
    if len(dct_subc[key]) < 19:
        print('short  ', key, "  it's length is", len(dct_subc[key]))
        todel += [key]       

todel = sorted(set(todel))
print(' ')
print('needed to be removed', todel)


#removing problematic subject(s) from dictionaries
for d in todel:
    del dct_thick[d]
    del dct_area[d]
    del dct_subc[d]
    del dct_tvol[d]

thickness
short   sub-NDAREE448KZ7Age099MonthsStudyYear01   it's length is 147
areas
short   sub-NDAREE448KZ7Age099MonthsStudyYear01   it's length is 147
subcortex
 
need to be removed ['sub-NDAREE448KZ7Age099MonthsStudyYear01']


In [42]:
#transform dct to table
df_thick = pd.DataFrame(dct_thick)
df_area = pd.DataFrame(dct_area)
df_subc = pd.DataFrame(dct_subc)
df_tvol = pd.DataFrame(dct_tvol)

#change indexes
df_thick.index = fs_cort_ind
df_area.index = fs_cort_ind
df_subc.index = fs_subc_ind_wb

#save tables
df_thick.T.to_csv(path_out + 'cortical_thickness.csv')
df_area.T.to_csv(path_out + 'cortical_area.csv')
df_subc.T.to_csv(path_out + 'subcortical_volume.csv')
df_tvol.T.to_csv(path_out + 'total_brain_volume.csv')